# search-cost

**How much cheaper is informed search?** BFS, Dijkstra and A\* over the same
queries on the same graph, measured rather than asserted.

The project catalogue states the problem as: *"show how informed search (A\*)
expands fewer nodes than uninformed search (BFS/Dijkstra) while returning the
same answer."* This notebook is where that claim gets its numbers.

It produces two required outputs:

1. the **BFS-vs-Dijkstra-vs-A\* comparison table** on long-haul queries, and
2. the **runtime-vs-input-size plot**, by narrowing the world network down a
   size series and re-timing the same queries at each size.

The data is pinned by `experiment.toml` and verified on load: if a byte of the
snapshot changes, this notebook raises instead of quietly producing a different
answer.

## Setup

Only the first cell differs between Colab and a local checkout.

In [ ]:
# In Colab, clone the repository and install the package first:
#   !git clone https://github.com/dgwartney/traiectoria-optima.git
#   %pip install -q ./traiectoria-optima matplotlib
# Then open this notebook from the clone.
try:
    import flight_planner  # noqa: F401
except ModuleNotFoundError as error:
    raise SystemExit('install the package first -- see the comment above') from error

In [ ]:
import statistics
import time
from pathlib import Path

from flight_planner.experiments import Experiment
from flight_planner.pathfinding import AStar, BFS, Dijkstra, ExpansionTrace

import plots

# The notebook lives in the experiment directory, so the experiment is
# right here. Nothing resolves against a repository root.
experiment = Experiment.open(Path.cwd())
parameters = experiment.parameters

PAIRS = [tuple(pair) for pair in parameters['pairs']]
REPEATS = parameters['repeats']
PAIRS, REPEATS

## The data

Opening the snapshot re-hashes every file against the manifest. This experiment
runs on the **whole world network**, unnarrowed — the catalogue asks for
long-haul queries, and those only exist if the graph has long hauls in it.

In [ ]:
snapshot = experiment.snapshot
print(snapshot.snapshot_id, snapshot.criteria or '(no criteria: the full network)')

catalog = experiment.catalog()
world = catalog.planner()
print(f'{len(catalog.airports):,} airports / {len(catalog.routes):,} routes')

## The three algorithms

One heuristic, used by A\* throughout: great-circle distance to the goal. It
never overestimates the remaining distance — no route between two airports is
shorter than the straight line between them — which is what makes it
admissible, and therefore what makes A\*'s answer optimal rather than merely
fast.

In [ ]:
def algorithms():
    """A fresh instance of each algorithm, in a fixed reporting order."""
    return [
        ('BFS', BFS()),
        ('Dijkstra', Dijkstra()),
        ('A*', AStar(lambda origin, goal: origin.distance_to(goal))),
    ]


NAMES = [name for name, _ in algorithms()]
NAMES

## 1. What each search costs

`search_route` returns the route *and* the counters the search accumulated:

- **expanded** — vertices whose outgoing edges were examined. This is the
  number the project's claim is about.
- **pushed** — vertices placed on the frontier, counting repeats. Both weighted
  algorithms re-queue a vertex rather than repositioning its entry, so
  `pushed - expanded` is the work the lazy-deletion design throws away.
- **peak** — the largest the frontier ever got, measured against the O(V) space
  bound the complexity write-up claims.

Note that **cost is not one unit**: BFS counts hops, the other two count
kilometres. They are never summed or compared directly.

In [ ]:
comparison = []
for origin, destination in PAIRS:
    row = {'pair': f'{origin}-{destination}',
           'expanded': {}, 'pushed': {}, 'peak': {}, 'cost': {}, 'legs': {}}
    for name, algorithm in algorithms():
        result = world.search_route(origin, destination, algorithm)
        row['expanded'][name] = result.nodes_expanded
        row['pushed'][name] = result.nodes_pushed
        row['peak'][name] = result.peak_frontier
        row['cost'][name] = result.cost
        row['legs'][name] = len(result.path)
    comparison.append(row)

header = f"{'pair':<10}{'algorithm':<10}{'expanded':>9}{'pushed':>8}{'peak':>7}{'cost':>13}{'legs':>6}"
print(header)
print('-' * len(header))
for row in comparison:
    for name in NAMES:
        unit = 'hops' if name == 'BFS' else 'km'
        print(f"{row['pair']:<10}{name:<10}{row['expanded'][name]:>9,}"
              f"{row['pushed'][name]:>8,}{row['peak'][name]:>7,}"
              f"{row['cost'][name]:>10,.0f} {unit:<5}{row['legs'][name]:>4}")
    print()

### Do Dijkstra and A\* actually agree?

The claim has two halves, and the second is worthless without the first. If A\*
were merely fast it would be a worse algorithm, not a better one.

In [ ]:
for row in comparison:
    agree = abs(row['cost']['Dijkstra'] - row['cost']['A*']) < 1e-9
    ratio = row['expanded']['Dijkstra'] / max(row['expanded']['A*'], 1)
    print(f"{row['pair']:<10} same distance: {str(agree):<6} "
          f"A* expanded {ratio:,.0f}x fewer nodes")

In [ ]:
plots.nodes_expanded(comparison, '../../slides/images/nodes-expanded.png', mode='dark')
plots.nodes_expanded(comparison, '../../docs/images/nodes-expanded-light.png', mode='light')

## 2. Where each search looks

The counters say *how much*. An `ExpansionTrace` says *where* — it records each
expansion as it happens, so the two searches can be compared by shape rather
than by size.

In [ ]:
origin, destination = PAIRS[0]
for name, algorithm in algorithms():
    if name == 'BFS':
        continue
    trace = ExpansionTrace()
    world.search_route(origin, destination, algorithm, observer=trace)
    looked = ' '.join(airport.iata_code for airport in trace.order[:14])
    tail = ' ...' if len(trace.order) > 14 else ''
    print(f'{name:<10} {len(trace.order):>4} expansions: {looked}{tail}')

## 3. Runtime against input size

The rubric asks for at least one plot of time against input size. Sizes come
from narrowing the world catalog with the vocabulary the `Catalog` already has,
so no new code decides what "smaller" means.

Two decisions worth stating, because both are easy to get wrong:

- **The x-axis is V + E, not airports.** The two do not move together — the US
  large-airport network is 94 airports but 7,005 routes, while Delta's is 427
  airports and 2,170 routes. Ordering by airport count produces a curve that
  crosses itself. V + E is also the term in O((V + E) log V).
- **The same queries run at every size.** Every airport in `PAIRS` survives
  every narrowing below. If a pair vanished partway down the series, the curve
  would be comparing different questions at different sizes.

In [ ]:
def narrow(base, spec):
    """Apply one size specification from experiment.toml to the catalog."""
    result = base
    if 'airline' in spec:
        result = result.airline(*spec['airline'])
    if 'country' in spec:
        result = result.country(*spec['country'])
    if 'airport_type' in spec:
        result = result.airport_type(*spec['airport_type'])
    return result


def median_ms(planner, algorithm):
    """Median query time in ms across PAIRS, after a discarded warm-up."""
    per_pair = []
    for origin, destination in PAIRS:
        planner.search_route(origin, destination, algorithm)   # warm-up
        runs = []
        for _ in range(REPEATS):
            started = time.perf_counter()
            # No observer while timing: measure the algorithm, not the watching.
            planner.search_route(origin, destination, algorithm)
            runs.append((time.perf_counter() - started) * 1000)
        per_pair.append(statistics.median(runs))
    return statistics.median(per_pair)

In [ ]:
series = []
for spec in parameters['sizes']:
    narrowed = narrow(catalog, spec)
    planner = narrowed.planner()          # built once, outside the timed region
    vertices, edges = len(narrowed.airports), len(narrowed.routes)
    series.append({
        'label': spec['label'],
        'vertices': vertices,
        'edges': edges,
        'size': vertices + edges,
        'median_ms': {name: median_ms(planner, algorithm)
                      for name, algorithm in algorithms()},
    })

header = f"{'narrowing':<18}{'V':>7}{'E':>9}{'V+E':>9}" + ''.join(f'{n:>12}' for n in NAMES)
print(header)
print('-' * len(header))
for row in sorted(series, key=lambda r: r['size']):
    print(f"{row['label']:<18}{row['vertices']:>7,}{row['edges']:>9,}{row['size']:>9,}"
          + ''.join(f"{row['median_ms'][n]:>12.3f}" for n in NAMES))

In [ ]:
plots.runtime_vs_size(series, '../../slides/images/runtime.png', mode='dark')
plots.runtime_vs_size(series, '../../docs/images/runtime-light.png', mode='light')

## Record

`experiment.record` writes `results.json` next to this notebook, carrying the
snapshot's identity, criteria and source commit alongside the numbers — so a
result can always be traced to the data that produced it.

In [ ]:
path = experiment.record({
    'comparison': comparison,
    'runtime_series': series,
    'agreement': [
        {'pair': row['pair'],
         'dijkstra_km': row['cost']['Dijkstra'],
         'astar_km': row['cost']['A*'],
         'expansion_ratio': row['expanded']['Dijkstra'] / max(row['expanded']['A*'], 1)}
        for row in comparison
    ],
}, catalog=catalog)
print(f'recorded -> {path}')

## What this shows

On the full world network, A\* returns **exactly** Dijkstra's distance on every
query while expanding orders of magnitude fewer nodes — the result the project
set out to demonstrate.

Two things worth carrying into the write-up because they complicate the simple
story:

- **BFS is not uniformly cheaper than Dijkstra.** It answers a different
  question (fewest hops, not shortest distance) and on some queries it expands
  *more* nodes than Dijkstra does while returning a longer route.
- **A\* pushes far more than it expands.** Its saving is in expansions — the
  expensive operation, since each one touches every outgoing edge — not in heap
  traffic. The `pushed` column is what makes that visible, and it is the honest
  answer to "is A\* really doing less work?"